In [ ]:
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

CODE_CHANGES_DIR = Path("../outputs/code_changes")

ok_records = [
    json.loads(p.read_text())
    for p in sorted(CODE_CHANGES_DIR.glob("*.json"))
    if not json.loads(p.read_text()).get("error")
]
print(f"Valid code-changes records: {len(ok_records)}")

In [ ]:
LANG_EXT = {
    "JavaScript/TypeScript": {".js", ".jsx", ".mjs", ".cjs", ".ts", ".tsx"},
    "Python":       {".py", ".pyw", ".pyx", ".pxd"},
    "Java":         {".java"},
    "C/C++":        {".c", ".h", ".cpp", ".cc", ".cxx", ".hpp", ".hh", ".hxx"},
    "PHP":          {".php", ".php3", ".php4", ".php5", ".phtml"},
    "Go":           {".go"},
    "Ruby":         {".rb", ".rake", ".ru", ".gemspec"},
    "Rust":         {".rs"},
    "C#":           {".cs"},
    "Kotlin":       {".kt", ".kts"},
    "Swift":        {".swift"},
    "Scala":        {".scala", ".sc"},
    "Shell":        {".sh", ".bash", ".zsh", ".fish"},
    "Perl":         {".pl", ".pm"},
    "Elixir":       {".ex", ".exs"},
    "Haskell":      {".hs", ".lhs"},
    "Dart":         {".dart"},
    "Lua":          {".lua"},
    "R":            {".r"},
    "Objective-C":  {".m", ".mm"},
}

EXT_TO_LANG = {ext: lang for lang, exts in LANG_EXT.items() for ext in exts}


def classify(files_list: list) -> set:
    langs = set()
    for f in files_list:
        lang = EXT_TO_LANG.get(Path(f).suffix.lower())
        if lang:
            langs.add(lang)
    if files_list and not langs:
        langs.add("Others")
    return langs


def cve_year(cve_id: str) -> int | None:
    m = re.match(r"CVE-(\d{4})-", cve_id)
    return int(m.group(1)) if m else None


def diff_files(record: dict) -> list:
    return [
        f
        for p in record.get("patches", [])
        if p.get("diff")
        for f in p.get("files", [])
    ]


print(f"Languages defined: {len(LANG_EXT)} + Others")
print(f"Total extensions mapped: {len(EXT_TO_LANG)}")

In [ ]:
lang_counter      = Counter()
year_lang_counter = defaultdict(Counter)
cve_lang_count    = []

total_classified = 0
no_diff_cves     = 0
no_ext_cves      = 0

for r in ok_records:
    year  = cve_year(r.get("cve_id", ""))
    files = diff_files(r)
    langs = classify(files)

    if not any(p.get("diff") for p in r.get("patches", [])):
        no_diff_cves += 1
        continue
    if not langs:
        no_ext_cves += 1
        continue

    total_classified += 1
    cve_lang_count.append(len(langs))
    for lang in langs:
        lang_counter[lang] += 1
        if year:
            year_lang_counter[year][lang] += 1

print(f"CVEs classified:               {total_classified}")
print(f"Skipped (all diffs null):      {no_diff_cves}")
print(f"Skipped (no known extension):  {no_ext_cves}")
print()

# Cross-reference with when.ipynb
import json as _json
from pathlib import Path as _Path
WHEN_DIR = _Path("../outputs/when")
no_date_in_when = {
    p.stem for p in WHEN_DIR.glob("*.json")
    if (rec := _json.loads(p.read_text())) and
       (rec.get("error") or rec.get("delta_days") is None)
}
no_diff_in_cc = {r["cve_id"] for r in ok_records
                 if all(p.get("diff") is None for p in r.get("patches", []))}
print("=== Cross-reference: skipped-here vs no-date in when.ipynb ===")
print(f"  code_changes: {len(no_diff_in_cc)} all-null | when: {len(no_date_in_when)} no-date")
print(f"  Both failed (no date AND no diff):              {len(no_date_in_when & no_diff_in_cc)}")
print(f"  No date in when BUT has diff here (tag API gap): {len(no_date_in_when - no_diff_in_cc)}  → these ARE classified above")
print(f"  Has date in when BUT no diff here:               {len(no_diff_in_cc - no_date_in_when)}  → these ARE skipped above")

In [ ]:
col_lang  = "Language"
col_cves  = "CVEs"
col_share = "Share"
print(f"{col_lang:<25} {col_cves:>6}  {col_share:>7}")
print("-" * 42)
for lang, cnt in lang_counter.most_common():
    print(f"{lang:<25} {cnt:>6}  {cnt/total_classified:>7.1%}")

In [ ]:
dist = Counter(cve_lang_count)
print("Languages per CVE:")
cutoff = 7
for n in sorted(k for k in dist if k <= cutoff):
    pct = dist[n] / total_classified
    print(f"  {n} language(s): {dist[n]:>5}  ({pct:.1%})")
rest = sum(v for k, v in dist.items() if k > cutoff)
if rest:
    print(f"  8+  language(s): {rest:>5}  ({rest/total_classified:.1%})")

In [ ]:
TOP = [
    "C/C++", "PHP", "JavaScript/TypeScript", "Python",
    "Go", "Java", "Ruby", "Rust", "Others",
]
YEARS = [y for y in sorted(year_lang_counter) if 2008 <= y <= 2026]

labels = ["Year"] + [l[:10] for l in TOP] + ["Total"]
widths = [6] + [12] * len(TOP) + [8]
header = "".join(f"{lbl:>{w}}" for lbl, w in zip(labels, widths))
print(header)
print("-" * len(header))
for y in YEARS:
    yc = year_lang_counter[y]
    row_total = sum(yc.get(l, 0) for l in TOP)
    vals = [str(y)] + [str(yc.get(l, 0)) for l in TOP] + [str(row_total)]
    print("".join(f"{v:>{w}}" for v, w in zip(vals, widths)))

In [ ]:
labels2 = ["Year"] + [l[:10] for l in TOP]
header2 = "".join(f"{lbl:>{w}}" for lbl, w in zip(labels2, [6] + [12]*len(TOP)))
print(header2)
print("-" * len(header2))
for y in YEARS:
    yc = year_lang_counter[y]
    total_y = sum(yc.values())
    if total_y == 0: continue
    vals = [str(y)] + [f"{yc.get(l, 0)/total_y:.1%}" for l in TOP]
    print("".join(f"{v:>{w}}" for v, w in zip(vals, [6] + [12]*len(TOP))))